# Model Training & Evaluation Pipeline - Karachi AQI Predictor
**Notebook**: `notebooks/model_training.ipynb`  
**Dataset**: `data/processed/karachi_selected_features.csv` (18 optimal features selected from feature selection stage)  
**Target Variable**: `target_pm25` (Next-Hour PM2.5 Concentration, $t+1$)

---

### Key Workflow:
1. **Load Selected Features**: Load `karachi_selected_features.csv` and inspect temporal attributes.
2. **Chronological Train/Test Split**: Perform an 80/20 time-based split to avoid data leakage.
3. **Candidate ML Model Experimentation**:
   - Linear Regression (Baseline)
   - Ridge Regression (L2 Regularization)
   - Lasso Regression (L1 Regularization)
   - Random Forest Regressor (Ensemble Bagging)
   - Gradient Boosting Regressor (Ensemble Boosting)
   - XGBoost Regressor (Extreme Gradient Boosting)
4. **Performance Evaluation**: Compare RMSE, MAE, and $R^2$ metrics on both train and test splits to monitor overfitting.
5. **Visualization**:
   - Model Leaderboard Bar Charts (RMSE, MAE, $R^2$)
   - Actual vs. Predicted Time Series & Scatter Plots
   - Residual Distribution & Quantile-Quantile (Q-Q) plots
   - Feature Importance / Coefficient rankings
6. **Hyperparameter Tuning**: Optimize champion model using TimeSeriesSplit Cross-Validation.
7. **Model Registry Export**: Save model artifact, scaler, metadata, and feature schema to `models/` directory.

## 1. Environment & Library Setup

In [ ]:
import os
import sys
import time
import json
import warnings
from pathlib import Path

warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# ML Models & Metrics
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from xgboost import XGBRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.model_selection import TimeSeriesSplit, RandomizedSearchCV
import joblib

# Formatting
%matplotlib inline
plt.style.use("seaborn-v0_8-whitegrid" if "seaborn-v0_8-whitegrid" in plt.style.available else "default")
plt.rcParams["figure.figsize"] = (12, 6)
plt.rcParams["font.size"] = 11
plt.rcParams["axes.titlesize"] = 13
pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda x: "%.4f" % x)

## 2. Load Selected Feature Dataset

In [ ]:
# Locate data path
DATA_PATH = Path("../data/processed/karachi_selected_features.csv") if Path("../data/processed/karachi_selected_features.csv").exists() else Path("data/processed/karachi_selected_features.csv")
print(f"Loading data from: {DATA_PATH.resolve()}")

df = pd.read_csv(DATA_PATH, parse_dates=["datetime"])
df = df.sort_values("datetime").reset_index(drop=True)

print(f"Dataset Shape       : {df.shape[0]:,} rows x {df.shape[1]} columns")
print(f"Datetime Coverage   : {df['datetime'].min()} to {df['datetime'].max()}")
print(f"Total Missing Values: {df.isnull().sum().sum()}")
df.head()

## 3. Separate Predictor Matrix ($X$) and Target ($y$)

In [ ]:
df_indexed = df.set_index("datetime")

TARGET_COL = "target_pm25"
y = df_indexed[TARGET_COL]
X = df_indexed.drop(columns=[TARGET_COL])

feature_names = list(X.columns)
print(f"Target variable: '{TARGET_COL}'")
print(f"Total Selected Predictors ({len(feature_names)}):")
for i, f in enumerate(feature_names, 1):
    print(f"  {i:2d}. {f}")

## 4. Time-Series Train/Test Split (80% Train, 20% Test)
Because air quality data is chronological, a standard random shuffle would cause severe look-ahead leakage. We use strict chronological splitting.

In [ ]:
TEST_RATIO = 0.20
split_idx = int(len(X) * (1 - TEST_RATIO))

X_train, X_test = X.iloc[:split_idx], X.iloc[split_idx:]
y_train, y_test = y.iloc[:split_idx], y.iloc[split_idx:]

print("Chronological Train / Test Split:")
print(f"  Train Set: {len(X_train):,} rows ({X_train.index.min()}  -->  {X_train.index.max()})")
print(f"  Test Set : {len(X_test):,} rows ({X_test.index.min()}  -->  {X_test.index.max()})")

# Feature Scaling (Fit on Train, Transform Test)
scaler = StandardScaler()
X_train_scaled = pd.DataFrame(scaler.fit_transform(X_train), columns=feature_names, index=X_train.index)
X_test_scaled = pd.DataFrame(scaler.transform(X_test), columns=feature_names, index=X_test.index)
print("[OK] Feature scaling completed.")

## 5. Candidate Regression Models Configuration

In [ ]:
models_config = {
    "Linear Regression": {
        "model": LinearRegression(),
        "scaled": True
    },
    "Ridge Regression": {
        "model": Ridge(alpha=1.0, random_state=42),
        "scaled": True
    },
    "Lasso Regression": {
        "model": Lasso(alpha=0.01, random_state=42, max_iter=2000),
        "scaled": True
    },
    "Random Forest": {
        "model": RandomForestRegressor(n_estimators=100, max_depth=14, min_samples_split=4, random_state=42, n_jobs=-1),
        "scaled": False
    },
    "Gradient Boosting": {
        "model": GradientBoostingRegressor(n_estimators=100, max_depth=5, learning_rate=0.08, random_state=42),
        "scaled": False
    },
    "XGBoost": {
        "model": XGBRegressor(n_estimators=120, max_depth=6, learning_rate=0.08, subsample=0.85, random_state=42, n_jobs=-1),
        "scaled": False
    }
}

print(f"Configured {len(models_config)} candidate regression models.")

## 6. Train and Evaluate All Models

In [ ]:
def calc_metrics(y_true, y_pred):
    mae = mean_absolute_error(y_true, y_pred)
    mse = mean_squared_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_true, y_pred)
    return {"MAE": mae, "RMSE": rmse, "R2": r2}

leaderboard = []
predictions_dict = {}

print("Training candidate models...")
for name, cfg in models_config.items():
    t0 = time.time()
    model = cfg["model"]
    is_scaled = cfg["scaled"]
    
    # Train
    X_tr = X_train_scaled if is_scaled else X_train
    X_te = X_test_scaled if is_scaled else X_test
    
    model.fit(X_tr, y_train)
    fit_time = time.time() - t0
    
    # Predict
    y_tr_pred = model.predict(X_tr)
    y_te_pred = model.predict(X_te)
    
    tr_metrics = calc_metrics(y_train, y_tr_pred)
    te_metrics = calc_metrics(y_test, y_te_pred)
    
    leaderboard.append({
        "Model": name,
        "Train_RMSE": tr_metrics["RMSE"],
        "Test_RMSE": te_metrics["RMSE"],
        "Train_MAE": tr_metrics["MAE"],
        "Test_MAE": te_metrics["MAE"],
        "Train_R2": tr_metrics["R2"],
        "Test_R2": te_metrics["R2"],
        "Overfit_R2_Gap": tr_metrics["R2"] - te_metrics["R2"],
        "Fit_Time_s": fit_time
    })
    
    predictions_dict[name] = y_te_pred

leaderboard_df = pd.DataFrame(leaderboard).sort_values("Test_RMSE").reset_index(drop=True)
leaderboard_df["Rank"] = leaderboard_df.index + 1

print("=== MODEL LEADERBOARD ===")
display(leaderboard_df[["Rank", "Model", "Test_RMSE", "Test_MAE", "Test_R2", "Train_R2", "Overfit_R2_Gap", "Fit_Time_s"]])

## 7. Model Performance Comparison Visualizations

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# 1. Test RMSE
sns.barplot(data=leaderboard_df, x="Model", y="Test_RMSE", ax=axes[0], palette="Reds_r")
axes[0].set_title("Test RMSE (Lower is Better)", fontweight="bold")
axes[0].tick_params(axis="x", rotation=30)
axes[0].set_ylabel("RMSE (ug/m3)")

# 2. Test MAE
sns.barplot(data=leaderboard_df, x="Model", y="Test_MAE", ax=axes[1], palette="Blues_r")
axes[1].set_title("Test MAE (Lower is Better)", fontweight="bold")
axes[1].tick_params(axis="x", rotation=30)
axes[1].set_ylabel("MAE (ug/m3)")

# 3. Test R2
sns.barplot(data=leaderboard_df, x="Model", y="Test_R2", ax=axes[2], palette="Greens_r")
axes[2].set_title("Test R² Score (Higher is Better)", fontweight="bold")
axes[2].tick_params(axis="x", rotation=30)
axes[2].set_ylabel("R² Score")

plt.tight_layout()
plt.show()

## 8. Actual vs. Predicted Predictions (Holdout Test Period)

In [ ]:
champion_name = leaderboard_df.iloc[0]["Model"]
champ_preds = predictions_dict[champion_name]

# Plot last 300 test hours for clear visual inspection
window = 300
plot_dates = y_test.index[-window:]
actual_vals = y_test.iloc[-window:].values
pred_vals = champ_preds[-window:]

plt.figure(figsize=(16, 6))
plt.plot(plot_dates, actual_vals, label="Actual Next-Hour PM2.5", color="#2c3e50", linewidth=2.0)
plt.plot(plot_dates, pred_vals, label=f"Predicted by {champion_name}", color="#e74c3c", linestyle="--", linewidth=1.8)

plt.title(f"Next-Hour PM2.5 Forecast vs Ground Truth ({champion_name} Champion Model)", fontsize=14, fontweight="bold", pad=15)
plt.xlabel("Datetime", fontsize=12)
plt.ylabel("PM2.5 (ug/m3)", fontsize=12)
plt.legend(frameon=True, fontsize=12, loc="upper right")
plt.tight_layout()
plt.show()

## 9. Residual Diagnostics & Parity Plot

In [ ]:
residuals = y_test.values - champ_preds

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))

# Parity scatter
ax1.scatter(y_test.values, champ_preds, alpha=0.25, color="#2980b9", edgecolors="none")
min_val = min(y_test.min(), champ_preds.min())
max_val = max(y_test.max(), champ_preds.max())
ax1.plot([min_val, max_val], [min_val, max_val], "r--", linewidth=2, label="Perfect 1:1 Line")
ax1.set_title(f"Parity Plot: Actual vs Predicted ({champion_name})", fontweight="bold")
ax1.set_xlabel("Actual PM2.5 (ug/m3)")
ax1.set_ylabel("Predicted PM2.5 (ug/m3)")
ax1.legend()

# Residual distribution
sns.histplot(residuals, kde=True, ax=ax2, color="#8e44ad", bins=40)
ax2.axvline(0, color="red", linestyle="--", linewidth=1.5)
ax2.set_title("Residual Distribution (Prediction Errors)", fontweight="bold")
ax2.set_xlabel("Residual (Actual - Predicted)")

plt.tight_layout()
plt.show()

## 10. Champion Model Feature Importance

In [ ]:
champ_model_obj = models_config[champion_name]["model"]

if hasattr(champ_model_obj, "feature_importances_"):
    feat_imp = pd.DataFrame({
        "Feature": feature_names,
        "Importance": champ_model_obj.feature_importances_
    }).sort_values("Importance", ascending=False)
    
    plt.figure(figsize=(10, 8))
    sns.barplot(data=feat_imp, y="Feature", x="Importance", palette="viridis")
    plt.title(f"Feature Importance - {champion_name}", fontsize=13, fontweight="bold", pad=15)
    plt.xlabel("Importance Weight")
    plt.ylabel("Feature")
    plt.tight_layout()
    plt.show()
else:
    print(f"{champion_name} does not expose feature_importances_.")

## 11. Hyperparameter Optimization on Champion Model

In [ ]:
print(f"Optimizing hyperparameters for {champion_name} with TimeSeriesSplit CV...")

tscv = TimeSeriesSplit(n_splits=4)

if champion_name == "XGBoost":
    param_dist = {
        "n_estimators": [100, 150, 200],
        "max_depth": [4, 5, 6, 7],
        "learning_rate": [0.03, 0.06, 0.1],
        "subsample": [0.8, 0.9, 1.0],
        "colsample_bytree": [0.8, 0.9, 1.0]
    }
    base_model = XGBRegressor(random_state=42, n_jobs=-1)
elif champion_name == "Random Forest":
    param_dist = {
        "n_estimators": [100, 150, 200],
        "max_depth": [10, 14, 18],
        "min_samples_split": [2, 4, 8],
        "min_samples_leaf": [1, 2, 4]
    }
    base_model = RandomForestRegressor(random_state=42, n_jobs=-1)
else:
    param_dist = {"alpha": [0.01, 0.1, 1.0, 10.0, 50.0]}
    base_model = Ridge(random_state=42)

search = RandomizedSearchCV(
    estimator=base_model,
    param_distributions=param_dist,
    n_iter=10,
    cv=tscv,
    scoring="neg_root_mean_squared_error",
    random_state=42,
    n_jobs=-1
)

search.fit(X_train, y_train)

best_tuned_model = search.best_estimator_
tuned_preds = best_tuned_model.predict(X_test)
tuned_metrics = calc_metrics(y_test, tuned_preds)

print(f"Best CV RMSE: {-search.best_score_:.4f}")
print("Best Hyperparameters:", search.best_params_)
print(f"Tuned Test Performance -> RMSE: {tuned_metrics['RMSE']:.4f} | MAE: {tuned_metrics['MAE']:.4f} | R²: {tuned_metrics['R2']:.4f}")

## 12. Save & Register Champion Model to Model Registry

In [ ]:
# Save champion model to local models/ registry
output_models_dir = Path("../models") if Path("../models").exists() else Path("models")
output_models_dir.mkdir(parents=True, exist_ok=True)

champion_file = output_models_dir / "best_model.pkl"
scaler_file = output_models_dir / "scaler.pkl"
features_file = output_models_dir / "selected_features.json"
meta_file = output_models_dir / "best_model_metadata.json"

# 1. Save model
joblib.dump(best_tuned_model, champion_file)

# 2. Save scaler
joblib.dump(scaler, scaler_file)

# 3. Save feature list
with open(features_file, "w") as f:
    json.dump(feature_names, f, indent=2)

# 4. Save metadata
metadata = {
    "champion_model": champion_name,
    "metrics": tuned_metrics,
    "num_features": len(feature_names),
    "features": feature_names,
    "best_hyperparameters": search.best_params_,
    "saved_at": pd.Timestamp.now().isoformat()
}
with open(meta_file, "w") as f:
    json.dump(metadata, f, indent=2)

print(f"[OK] Champion Model ({champion_name}) saved successfully to {output_models_dir.resolve()}")
print(f"[OK] Saved model: {champion_file.name}")
print(f"[OK] Saved scaler: {scaler_file.name}")
print(f"[OK] Saved metadata: {meta_file.name}")